# Lab RAG para Designing Machine Learning Systems

Este notebook inicia a linha de base (baseline RAG) e a variante adaptive RAG para comparar:
- qualidade das respostas
- latência
- custo de chamadas ao LLM
- variação entre execuções

A estrutura foi desenhada para usar o PDF local do livro e as 20 perguntas em `questions.json`.


In [32]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd()
CONFIG_PATH = ROOT / 'configs' / 'experiment.json'
QUESTIONS_PATH = ROOT / 'questions.json'

def load_json(path):
    with open(path, 'r', encoding='utf-8') as fh:
        return json.load(fh)

config = load_json(CONFIG_PATH)
questions = load_json(QUESTIONS_PATH)
print(f"Projeto: {config.get('project_name', 'rag-dmls-lab')}")
print(f"Perguntas carregadas: {len(questions.get('questions', []))}")
print(f"Tema: {config.get('topic', 'Designing Machine Learning Systems')}")


Projeto: rag-dmls-lab
Perguntas carregadas: 20
Tema: Designing Machine Learning Systems


In [33]:
def find_pdf_path(root):
    candidates = [root / 'corpus' / 'documentos', root / 'corpus']
    for folder in candidates:
        if folder.exists():
            matches = sorted(folder.rglob('*.pdf'))
            if matches:
                return matches[0]
    raise FileNotFoundError(
        'PDF do livro não encontrado. Coloque a cópia em corpus/documentos/ ou corpus/.'
    )

def read_pdf_pages(pdf_path):
    reader = PdfReader(str(pdf_path))
    pages = []
    for page in reader.pages:
        text = page.extract_text() or ''
        pages.append(text)
    return pages

def chunk_text(text, chunk_size=800, overlap=120):
    text = text.replace('\r', '\n')
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == len(text):
            break
        start = max(start + chunk_size - overlap, end - overlap)
    return chunks

pdf_path = find_pdf_path(ROOT)
print(f'Arquivo PDF localizado: {pdf_path}')


Arquivo PDF localizado: /mnt/ssd/projeto_rag/rag-dmls-lab/corpus/documentos/designing-machine-learning-systems-2022.pdf


In [34]:
def build_index(chunks):
    vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
    matrix = vectorizer.fit_transform(chunks)
    return vectorizer, matrix

def expand_query(query):
    expansions = {
        'business value': 'business objective success metric problem framing use case',
        'clear definition of the problem': 'problem framing task definition objective success criteria',
        'data-centric approach': 'data-centric AI data quality labels dataset improvement',
        'model-centric approach': 'model-centric AI algorithm architecture model improvement',
        'real-world performance': 'production performance deployment distribution shift noisy inputs',
        'feedback loops': 'degenerate feedback loops popularity bias production predictions',
    }
    lowered = query.lower()
    return ' '.join([query] + [value for key, value in expansions.items() if key in lowered])

def retrieve_top_k(query, chunks, vectorizer, matrix, k=4):
    query = expand_query(query)
    q_vector = vectorizer.transform([query])
    scores = cosine_similarity(q_vector, matrix).ravel()
    ranked = np.argsort(scores)[::-1]
    ranked = [idx for idx in ranked if scores[idx] > 0][:k]
    return [(chunks[idx], float(scores[idx])) for idx in ranked]

def format_prompt(question, context):
    return (
        'Answer the question using only the provided context. '
        'Answer in English. If the context does not contain enough evidence, say so explicitly.\n\n'
        f'Question: {question}\n\nContext:\n{context}\n\nAnswer:'
    )

def route_question(question):
    q = question.lower()
    adaptive_markers = [
        'compare', 'tradeoff', 'trade-off', 'architecture', 'design', 'monitor',
        'monitoring', 'latency', 'cost', 'observability', 'system', 'serving'
    ]
    return 'adaptive' if any(token in q for token in adaptive_markers) else 'baseline'

print('Funções de busca e roteamento prontas.')


Funções de busca e roteamento prontas.


In [35]:
OLLAMA_URL = 'http://localhost:11434/api/generate'

def call_ollama(prompt, model=None, temperature=None, timeout=120):
    model = model or config.get('generator_model', 'qwen2.5:7b')
    temperature = temperature if temperature is not None else config.get('generation_temperature', 0.0)
    payload = {
        'model': model,
        'prompt': prompt,
        'stream': False,
        'options': {'temperature': temperature},
    }
    response = requests.post(OLLAMA_URL, json=payload, timeout=timeout)
    response.raise_for_status()
    data = response.json()
    return data.get('response', '').strip()

def answer_question_baseline(question, chunks, vectorizer, matrix, k=None, model=None):
    k = k or config.get('top_k', 4)
    hits = retrieve_top_k(question, chunks, vectorizer, matrix, k=k)
    if not hits:
        return 'Não encontrei evidência suficiente no corpus para responder com confiança.'
    context = '\n\n---\n\n'.join([f'[Passage {i + 1}] {text[:1600]}' for i, (text, _) in enumerate(hits)])
    prompt = format_prompt(question, context)
    try:
        return call_ollama(prompt, model=model)
    except Exception as exc:
        return f'Erro ao chamar o Ollama: {exc}'

def answer_question_adaptive(question, chunks, vectorizer, matrix, k=None, model=None):
    k = k or config.get('top_k', 4)
    q = question.lower()
    variants = [expand_query(question)]
    if any(token in q for token in ['compare', 'tradeoff', 'trade-off', 'architecture', 'design', 'monitor', 'monitoring', 'latency', 'cost', 'observability', 'system', 'serving']):
        variants.extend([
            'trade-offs in designing machine learning systems',
            'monitoring, evaluation and operational constraints in ML',
            'data quality and deployment considerations in ML systems'
        ])
    context_parts = []
    seen = set()
    for variant in variants:
        for text, score in retrieve_top_k(variant, chunks, vectorizer, matrix, k=k):
            signature = text[:200]
            if signature not in seen:
                seen.add(signature)
                context_parts.append((text, score))
    if not context_parts:
        return 'Não encontrei evidência suficiente no corpus para responder com confiança.'
    context = '\n\n---\n\n'.join([f'[Passage {i + 1}] {text[:1600]}' for i, (text, _) in enumerate(context_parts[:k])])
    prompt = format_prompt(question, context)
    try:
        return call_ollama(prompt, model=model)
    except Exception as exc:
        return f'Erro ao chamar o Ollama: {exc}'

print('Chamada ao LLM e respostas de baseline/adaptive prontas.')


Chamada ao LLM e respostas de baseline/adaptive prontas.


In [36]:
def run_demo_experiment():
    raw_pages = read_pdf_pages(pdf_path)
    raw_text = '\n\n'.join(raw_pages)
    chunk_size = config.get('chunk_size_chars', 800)
    overlap = config.get('chunk_overlap_chars', 120)
    chunks = chunk_text(raw_text, chunk_size=chunk_size, overlap=overlap)
    vectorizer, matrix = build_index(chunks)
    dataset = questions.get('questions', [])
    results = []

    for item in dataset[:5]:
        question = item['question']
        start = time.time()
        route = route_question(question)
        answer = (
            answer_question_adaptive(question, chunks, vectorizer, matrix)
            if route == 'adaptive'
            else answer_question_baseline(question, chunks, vectorizer, matrix)
        )
        elapsed = time.time() - start
        results.append({
            'id': item['id'],
            'route': route,
            'latency_sec': round(elapsed, 3),
            'answer': answer
        })

    return pd.DataFrame(results)

sample_df = run_demo_experiment()
sample_df.head()


,id,route,latency_sec,answer
0,Q01,baseline,12.710,The book emphasizes starting with data quality...
1,Q02,adaptive,20.231,The book describes the debate between improvin...
2,Q03,adaptive,9.591,Building a simple baseline is essential before...
3,Q04,adaptive,19.126,Data quality problems such as label noise and ...
4,Q05,adaptive,15.097,Hidden feedback loops are degenerate feedback ...
